# Checkpoint B2: Hook đã cài đặt

Pre_tool_call hook đã tạo, cấp quyền, và tự động cấu hình vào profile. Tiếp tục từ **Bước B3: Kiểm thử hành vi**.

In [1]:
# Tự động hóa: Setup Knowledge Base + Copy Hook + Gắn Hook vào profile
import os
import shutil
import subprocess

# 1. Xác định thư mục templates nguồn
notebook_dir = os.getcwd()
possible_paths = [
    os.path.abspath(os.path.join(notebook_dir, "..")),
    os.path.abspath("."),
    os.path.abspath(".."),
    os.path.abspath("../../templates"),
]
template_dir = None
for p in possible_paths:
    if os.path.exists(os.path.join(p, "skills/vtn-agent-skill/kb/bgp-config-sample.md")):
        template_dir = os.path.join(p, "skills/vtn-agent-skill")
        break

if not template_dir:
    backup_dir = "/Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-03/session-05-mini-tool-vibe-coding/templates"
    if os.path.exists(backup_dir):
        template_dir = os.path.join(backup_dir, "skills/vtn-agent-skill")

# 2. Tự động hóa: Copy Knowledge Base vào thư mục project
project_kb_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "output", "vtn-session05", "templates", "skills", "vtn-agent-skill", "kb"))
os.makedirs(project_kb_dir, exist_ok=True)
print(f"📂 Đã chuẩn bị thư mục KB tại: {project_kb_dir}")

kb_files = ["bgp-config-sample.md", "incident-template.md", "kb-inventory.md"]
if template_dir:
    for f in kb_files:
        src = os.path.join(template_dir, "kb", f)
        dst = os.path.join(project_kb_dir, f)
        if os.path.exists(src):
            try:
                shutil.copy(src, dst)
                print(f"✅ Đã sao chép KB: {f}")
            except Exception as e:
                print(f"❌ Lỗi copy KB {f}: {e}")
        else:
            print(f"⚠️ Không tìm thấy KB nguồn: {src}")
else:
    print("❌ Không tìm thấy thư mục templates nguồn để copy KB.")

# 3. Tự động hóa: Copy Hook script và cấp quyền
hermes_hooks_dir = os.path.expanduser("~/.hermes/agent-hooks")
os.makedirs(hermes_hooks_dir, exist_ok=True)
hook_target = os.path.join(hermes_hooks_dir, "block-write-and-shell.py")

if template_dir:
    hook_src = os.path.join(template_dir, "scripts", "hook-block-write.py")
    if os.path.exists(hook_src):
        try:
            shutil.copy(hook_src, hook_target)
            os.chmod(hook_target, 0o755)
            print(f"✅ Đã cài đặt hook và cấp quyền thực thi: {hook_target}")
        except Exception as e:
            print(f"❌ Lỗi cài đặt hook: {e}")
    else:
        print(f"⚠️ Không tìm thấy hook nguồn: {hook_src}")

# 4. Tự động hóa: Gắn Hook vào profile tri-thuc-noi-bo
config_path = os.path.expanduser("~/.hermes/profiles/tri-thuc-noi-bo/config.yaml")
if os.path.exists(config_path):
    try:
        with open(config_path, "r", encoding="utf-8") as f:
            content = f.read()
        
        if "hooks:" not in content:
            hook_config = f"\nhooks:\n  pre_tool_call:\n    - matcher: \"write_file|patch|terminal|process|execute_code\"\n      command: \"{hook_target}\"\n      timeout: 5\nhooks_auto_accept: true\n"
            with open(config_path, "a", encoding="utf-8") as f:
                f.write(hook_config)
            print("✅ Đã tự động gắn cấu hình Hook vào profile config.yaml!")
        else:
            print("ℹ️ Cấu hình Hook đã có sẵn trong config.yaml.")
    except Exception as e:
        print(f"❌ Lỗi ghi cấu hình Hook: {e}")
else:
    print(f"⚠️ Chưa có file config.yaml tại {config_path}. Đảm bảo đã chạy Checkpoint A2.")

📂 Đã chuẩn bị thư mục KB tại: /Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-03/session-05-mini-tool-vibe-coding/output/vtn-session05/templates/skills/vtn-agent-skill/kb
✅ Đã sao chép KB: bgp-config-sample.md
✅ Đã sao chép KB: incident-template.md
✅ Đã sao chép KB: kb-inventory.md
✅ Đã cài đặt hook và cấp quyền thực thi: /Users/shimazu/.hermes/agent-hooks/block-write-and-shell.py
✅ Đã tự động gắn cấu hình Hook vào profile config.yaml!


In [2]:
# Kiểm thử hook trực tiếp với tool write_file (mong đợi: BLOCKED)
import subprocess
import json
import os

hook_path = os.path.expanduser("~/.hermes/agent-hooks/block-write-and-shell.py")

test_input_write = json.dumps({
    "hook_event_name": "pre_tool_call",
    "tool_name": "write_file",
    "tool_input": {
        "path": "/docs/simulated/test.md",
        "content": "test"
    }
})

if os.path.exists(hook_path):
    result = subprocess.run(
        ["python3", hook_path],
        input=test_input_write,
        capture_output=True,
        text=True,
        timeout=10
    )
    print("Test write_file:")
    print(f"  stdout: {result.stdout.strip()}")
    print(f"  stderr: {result.stderr.strip()}")
else:
    print("Hook chưa tồn tại. Kiểm tra lại Bước A2 hoặc chạy lại cell 1.")

Test write_file:
  stdout: {"action": "block", "message": "Lab hook active: blocked tool write_file. Agent is read-only per VTN security policy."}
  stderr: 


In [3]:
# Kiểm thử hook trực tiếp với tool terminal (mong đợi: BLOCKED)
import subprocess
import json
import os

hook_path = os.path.expanduser("~/.hermes/agent-hooks/block-write-and-shell.py")

test_input_terminal = json.dumps({
    "hook_event_name": "pre_tool_call",
    "tool_name": "terminal",
    "tool_input": {
        "command": "ls /docs/simulated"
    }
})

if os.path.exists(hook_path):
    result = subprocess.run(
        ["python3", hook_path],
        input=test_input_terminal,
        capture_output=True,
        text=True,
        timeout=10
    )
    print("Test terminal:")
    print(f"  stdout: {result.stdout.strip()}")
    print(f"  stderr: {result.stderr.strip()}")
else:
    print("Hook chưa tồn tại.")

Test terminal:
  stdout: {"action": "block", "message": "Lab hook active: blocked tool terminal. Agent is read-only per VTN security policy."}
  stderr: 


In [4]:
# Xác minh hook cấu hình trong config.yaml của 3 profiles
import os

profiles = ["tri-thuc-noi-bo", "soan-thao-noi-dung", "checklist-van-hanh"]
base = os.path.expanduser("~/.hermes/profiles")

for profile_name in profiles:
    config_path = os.path.join(base, profile_name, "config.yaml")
    print(f"Profile: {profile_name}")
    if os.path.exists(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            content = f.read()
        has_hook = "hook" in content.lower() or "pre_tool" in content.lower()
        status = "HOOK OK" if has_hook else "⚠️ CHƯA GẮN HOOK"
        print(f"  Trạng thái: {status}")
    else:
        print(f"  ⚠️ Chưa có config.yaml")
    print()

Profile: tri-thuc-noi-bo
  Trạng thái: HOOK OK

Profile: soan-thao-noi-dung
  Trạng thái: ⚠️ CHƯA GẮN HOOK

Profile: checklist-van-hanh
  Trạng thái: ⚠️ CHƯA GẮN HOOK



---

**Tiếp tục:** Quay lại `lab.md` → **Bước B3: Kiểm thử hành vi 3 Agent**